In [1]:
# KPI de ventes produit ?

from pathlib import Path

import duckdb
import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATABASE_FILE = (
    PROJECT_ROOT
    / "data"
    / "warehouse"
    / "customer_value_radar.duckdb"
)

connection = duckdb.connect(
    str(DATABASE_FILE)
)

print("Connexion DuckDB : OK")

Connexion DuckDB : OK


In [2]:
# Analyse des catégories Silver

category_summary = connection.sql(
    """
    SELECT
        line_category,
        COUNT(*) AS rows,
        SUM(
            CASE
                WHEN is_cancellation THEN 1
                ELSE 0
            END
        ) AS cancellation_rows,
        SUM(
            CASE
                WHEN Quantity > 0 THEN 1
                ELSE 0
            END
        ) AS positive_quantity_rows,
        SUM(
            CASE
                WHEN Quantity < 0 THEN 1
                ELSE 0
            END
        ) AS negative_quantity_rows,
        SUM(
            CASE
                WHEN Price > 0 THEN 1
                ELSE 0
            END
        ) AS positive_price_rows,
        ROUND(
            SUM(raw_line_value),
            2
        ) AS raw_value
    FROM silver_transactions
    GROUP BY line_category
    ORDER BY rows DESC
    """
).df()

display(category_summary)

,line_category,rows,cancellation_rows,positive_quantity_rows,negative_quantity_rows,positive_price_rows,raw_value
0,product,1039017,17974.0,1017670.0,21347.0,1033062.0,18981261.52
1,shipping,3788,238.0,3550.0,238.0,3771.0,433410.51
2,manual,1403,535.0,869.0,534.0,1396.0,-82935.57
3,discount,173,168.0,5.0,168.0,173.0,-12879.63
4,fee,152,115.0,37.0,115.0,152.0,-264936.18
5,sample,102,99.0,3.0,99.0,102.0,-6000.85
6,gift_voucher,100,1.0,99.0,1.0,78.0,1686.61
7,adjustment,76,31.0,45.0,31.0,71.0,-140047.79
8,stock_movement,20,0.0,0.0,20.0,0.0,0.00
9,test,17,4.0,13.0,4.0,14.0,203.50


In [4]:
# Affichage complet des résultats

print("CATEGORY SUMMARY")
print(
    category_summary.to_string(index=False)
)

print("\nPRODUCT SUMMARY")
print(
    product_summary.to_string(index=False)
)

CATEGORY SUMMARY
 line_category    rows  cancellation_rows  positive_quantity_rows  negative_quantity_rows  positive_price_rows   raw_value
       product 1039017            17974.0               1017670.0                 21347.0            1033062.0 18981261.52
      shipping    3788              238.0                  3550.0                   238.0               3771.0   433410.51
        manual    1403              535.0                   869.0                   534.0               1396.0   -82935.57
      discount     173              168.0                     5.0                   168.0                173.0   -12879.63
           fee     152              115.0                    37.0                   115.0                152.0  -264936.18
        sample     102               99.0                     3.0                    99.0                102.0    -6000.85
  gift_voucher     100                1.0                    99.0                     1.0                 78.0     1686.61

In [3]:
# Détail des lignes classées product

product_summary = connection.sql(
    """
    SELECT
        is_cancellation,
        CASE
            WHEN Quantity > 0 THEN 'positive'
            WHEN Quantity < 0 THEN 'negative'
            ELSE 'zero'
        END AS quantity_sign,
        CASE
            WHEN Price > 0 THEN 'positive'
            WHEN Price < 0 THEN 'negative'
            ELSE 'zero'
        END AS price_sign,
        COUNT(*) AS rows,
        ROUND(
            SUM(raw_line_value),
            2
        ) AS raw_value
    FROM silver_transactions
    WHERE line_category = 'product'
    GROUP BY
        is_cancellation,
        quantity_sign,
        price_sign
    ORDER BY rows DESC
    """
).df()

display(product_summary)

,is_cancellation,quantity_sign,price_sign,rows,raw_value
0,False,positive,positive,1015088,19700954.46
1,True,negative,positive,17974,-719692.94
2,False,negative,zero,3373,0.00
3,False,positive,zero,2582,0.00


In [5]:
# Validation du périmètre des ventes produit

product_scope = connection.sql(
    """
    SELECT
        SUM(
            CASE
                WHEN line_category = 'product'
                    AND is_cancellation = FALSE
                    AND Quantity > 0
                    AND Price > 0
                THEN 1
                ELSE 0
            END
        ) AS product_sale_rows,

        SUM(
            CASE
                WHEN line_category = 'product'
                    AND is_cancellation = TRUE
                    AND Quantity < 0
                    AND Price > 0
                THEN 1
                ELSE 0
            END
        ) AS product_cancellation_rows,

        ROUND(
            SUM(
                CASE
                    WHEN line_category = 'product'
                        AND is_cancellation = FALSE
                        AND Quantity > 0
                        AND Price > 0
                    THEN raw_line_value
                    ELSE 0
                END
            ),
            2
        ) AS gross_product_sales,

        ROUND(
            SUM(
                CASE
                    WHEN line_category = 'product'
                        AND is_cancellation = TRUE
                        AND Quantity < 0
                        AND Price > 0
                    THEN raw_line_value
                    ELSE 0
                END
            ),
            2
        ) AS product_cancellations,

        ROUND(
            SUM(
                CASE
                    WHEN line_category = 'product'
                        AND Price > 0
                    THEN raw_line_value
                    ELSE 0
                END
            ),
            2
        ) AS net_product_sales

    FROM silver_transactions
    """
).df()

display(product_scope)

,product_sale_rows,product_cancellation_rows,gross_product_sales,product_cancellations,net_product_sales
0,1015088.0,17974.0,19700954.46,-719692.94,18981261.52


## Les ventes produit

L'analyse de la couche Silver permet de distinguer les ventes produit , les annulations et les autres mouvements.

### Vente produit

Une ligne est une vente produit lorsque :

- `line_category = 'product'`
- `is_cancellation = False`
- `Quantity > 0`
- `Price > 0`

### Annulation produit

Une ligne est une annulation de vente produit lorsque :

- `line_category = 'product'`
- `is_cancellation = True`
- `Quantity < 0`
- `Price > 0`

### Lignes produit hors ventes payantes

Les lignes produit à prix nul sont conservées dans les données mais ne sont pas intégrées à la valeur des ventes produit.

### Indicateurs retenus

- `gross_product_sales` : valeur des ventes produit positives avant annulations ;
- `product_cancellations` : valeur signée des annulations produit ;
- `net_product_sales` : ventes produit positives après prise en compte des annulations.

Ces indicateurs constituent des métriques analytiques du projet. Le terme « valeur des ventes produit » est privilégié à ce stade plutôt que « chiffre d'affaires comptable ».

In [6]:
connection.close()

print("Connexion DuckDB fermée")

Connexion DuckDB fermée


## Validation de fact_order_lines

La table `fact_order_lines` conserve le grain d'une ligne transactionnelle et ajoute les indicateurs nécessaires aux analyses.

In [7]:
# Reconnexion à DuckDB

connection = duckdb.connect(
    str(DATABASE_FILE)
)

print("Connexion DuckDB : OK")

Connexion DuckDB : OK


In [8]:
# Aperçu de fact_order_lines

fact_order_lines_sample = connection.sql(
    """
    SELECT
        Invoice,
        StockCode,
        Description,
        Quantity,
        InvoiceDate,
        Price,
        "Customer ID",
        Country,
        line_category,
        is_cancellation,
        is_product_sale,
        is_product_cancellation,
        gross_product_sales_value,
        product_cancellation_value,
        net_product_sales_value
    FROM fact_order_lines
    LIMIT 20
    """
).df()

display(fact_order_lines_sample)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_category,is_cancellation,is_product_sale,is_product_cancellation,gross_product_sales_value,product_cancellation_value,net_product_sales_value
0,489436,22194,BLACK DINER WALL CLOCK,2,2009-12-01 09:06:00,8.50,13078,United Kingdom,product,False,True,False,17.00,0.0,17.00
1,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,2009-12-01 09:06:00,4.65,13078,United Kingdom,product,False,True,False,55.80,0.0,55.80
2,489438,84031A,CHARLIE+LOLA RED HOT WATER BOTTLE,56,2009-12-01 09:24:00,3.00,18102,United Kingdom,product,False,True,False,168.00,0.0,168.00
3,489438,84519B,CARROT CHARLIE+LOLA COASTER SET,60,2009-12-01 09:24:00,2.40,18102,United Kingdom,product,False,True,False,144.00,0.0,144.00
4,489442,22272,FELTCRAFT DOLL MARIA,6,2009-12-01 09:46:00,2.95,13635,United Kingdom,product,False,True,False,17.70,0.0,17.70
5,489446,20726,LUNCH BAG WOODLAND,10,2009-12-01 10:06:00,1.65,13758,United Kingdom,product,False,True,False,16.50,0.0,16.50
6,489446,85123A,WHITE HANGING HEART T-LIGHT HOLDER,32,2009-12-01 10:06:00,2.55,13758,United Kingdom,product,False,True,False,81.60,0.0,81.60
7,489450,21896,POTTING SHED TWINE,6,2009-12-01 10:36:00,2.10,16321,Australia,product,False,True,False,12.60,0.0,12.60
8,489460,22273,FELTCRAFT DOLL MOLLY,6,2009-12-01 10:46:00,2.95,16167,United Kingdom,product,False,True,False,17.70,0.0,17.70
9,489460,79323W,WHITE CHERRY LIGHTS,8,2009-12-01 10:46:00,6.75,16167,United Kingdom,product,False,True,False,54.00,0.0,54.00


In [9]:
#  par type de ligne

examples = connection.sql(
    """
    (
        SELECT *
        FROM fact_order_lines
        WHERE is_product_sale = TRUE
        LIMIT 5
    )

    UNION ALL

    (
        SELECT *
        FROM fact_order_lines
        WHERE is_product_cancellation = TRUE
        LIMIT 5
    )

    UNION ALL

    (
        SELECT *
        FROM fact_order_lines
        WHERE line_category <> 'product'
        LIMIT 10
    )
    """
).df()

display(
    examples[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_category",
            "is_cancellation",
            "is_product_sale",
            "is_product_cancellation",
            "net_product_sales_value"
        ]
    ]
)

,Invoice,StockCode,Description,Quantity,Price,line_category,is_cancellation,is_product_sale,is_product_cancellation,net_product_sales_value
0,489436,22194,BLACK DINER WALL CLOCK,2,8.50,product,False,True,False,17.00
1,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,4.65,product,False,True,False,55.80
2,489438,84031A,CHARLIE+LOLA RED HOT WATER BOTTLE,56,3.00,product,False,True,False,168.00
3,489438,84519B,CARROT CHARLIE+LOLA COASTER SET,60,2.40,product,False,True,False,144.00
4,489442,22272,FELTCRAFT DOLL MARIA,6,2.95,product,False,True,False,17.70
5,C489449,21871,SAVE THE PLANET MUG,-12,1.25,product,True,False,True,-15.00
6,C489518,20892,SET/3 TALL GLASS CANDLE HOLDER PINK,-2,12.75,product,True,False,True,-25.50
7,C489583,72802B,OCEAN SCENT CANDLE IN JEWELLED BOX,-2,4.25,product,True,False,True,-8.50
8,C489610,20682,RED SPOTTY CHILDS UMBRELLA,-1,3.25,product,True,False,True,-3.25
9,C489610,22138,BAKING SET 9 PIECE RETROSPOT,-2,4.95,product,True,False,True,-9.90


In [10]:
# Dimensions de la table

fact_shape = connection.sql(
    """
    SELECT
        COUNT(*) AS rows
    FROM fact_order_lines
    """
).df()

columns = connection.sql(
    """
    DESCRIBE fact_order_lines
    """
).df()

print("Nombre de lignes :", fact_shape.loc[0, "rows"])
print("Nombre de colonnes :", len(columns))

display(columns)

Nombre de lignes : 1044848
Nombre de colonnes : 24


,column_name,column_type,null,key,default,extra
0,Invoice,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,Price,DOUBLE,YES,None,None,None
6,Customer ID,BIGINT,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None
8,SourcePeriod,VARCHAR,YES,None,None,None
9,line_category,VARCHAR,YES,None,None,None


In [11]:
connection.close()

print("Connexion DuckDB fermée")

Connexion DuckDB fermée


In [12]:
connection = duckdb.connect(
    str(DATABASE_FILE)
)

print("Connexion DuckDB : OK")

Connexion DuckDB : OK


In [13]:
# Aperçu de fact_orders

fact_orders_sample = connection.sql(
    """
    SELECT *
    FROM fact_orders
    ORDER BY order_datetime
    LIMIT 20
    """
).df()

display(fact_orders_sample)

,Invoice,order_datetime,last_line_datetime,customer_id,country,source_period,line_count,distinct_stockcodes,total_quantity,raw_order_value,product_sale_quantity,product_cancellation_quantity,gross_product_sales_value,product_cancellation_value,net_product_sales_value,is_cancellation,has_product_sale,has_product_cancellation
0,489434,2009-12-01 07:45:00,2009-12-01 07:45:00,13085,United Kingdom,2009-2010,8,8,166.0,505.30,166.0,0.0,505.30,0.0,505.30,False,True,False
1,489435,2009-12-01 07:46:00,2009-12-01 07:46:00,13085,United Kingdom,2009-2010,4,4,60.0,145.80,60.0,0.0,145.80,0.0,145.80,False,True,False
2,489436,2009-12-01 09:06:00,2009-12-01 09:06:00,13078,United Kingdom,2009-2010,19,19,193.0,630.33,193.0,0.0,630.33,0.0,630.33,False,True,False
3,489437,2009-12-01 09:08:00,2009-12-01 09:08:00,15362,United Kingdom,2009-2010,23,23,145.0,310.75,145.0,0.0,310.75,0.0,310.75,False,True,False
4,489438,2009-12-01 09:24:00,2009-12-01 09:24:00,18102,United Kingdom,2009-2010,17,17,826.0,2286.24,826.0,0.0,2286.24,0.0,2286.24,False,True,False
5,489439,2009-12-01 09:28:00,2009-12-01 09:28:00,12682,France,2009-2010,19,19,219.0,426.30,216.0,0.0,372.30,0.0,372.30,False,True,False
6,489440,2009-12-01 09:43:00,2009-12-01 09:43:00,18087,United Kingdom,2009-2010,2,2,16.0,50.40,16.0,0.0,50.40,0.0,50.40,False,True,False
7,489441,2009-12-01 09:44:00,2009-12-01 09:44:00,18087,United Kingdom,2009-2010,4,4,102.0,344.34,102.0,0.0,344.34,0.0,344.34,False,True,False
8,489442,2009-12-01 09:46:00,2009-12-01 09:46:00,13635,United Kingdom,2009-2010,23,23,275.0,382.37,275.0,0.0,382.37,0.0,382.37,False,True,False
9,489443,2009-12-01 09:50:00,2009-12-01 09:50:00,14110,United Kingdom,2009-2010,7,7,120.0,285.06,120.0,0.0,285.06,0.0,285.06,False,True,False


In [14]:
# Structure de fact_orders

fact_orders_columns = connection.sql(
    """
    DESCRIBE fact_orders
    """
).df()

print("Nombre de lignes :", connection.sql(
    "SELECT COUNT(*) FROM fact_orders"
).fetchone()[0])

print("Nombre de colonnes :", len(fact_orders_columns))

display(fact_orders_columns)

Nombre de lignes : 53628
Nombre de colonnes : 18


,column_name,column_type,null,key,default,extra
0,Invoice,VARCHAR,YES,None,None,None
1,order_datetime,TIMESTAMP,YES,None,None,None
2,last_line_datetime,TIMESTAMP,YES,None,None,None
3,customer_id,BIGINT,YES,None,None,None
4,country,VARCHAR,YES,None,None,None
5,source_period,VARCHAR,YES,None,None,None
6,line_count,BIGINT,YES,None,None,None
7,distinct_stockcodes,BIGINT,YES,None,None,None
8,total_quantity,HUGEINT,YES,None,None,None
9,raw_order_value,DOUBLE,YES,None,None,None


In [15]:
# Exemples de commandes avec ventes produit

orders_examples = connection.sql(
    """
    SELECT
        Invoice,
        order_datetime,
        customer_id,
        country,
        line_count,
        distinct_stockcodes,
        product_sale_quantity,
        gross_product_sales_value,
        product_cancellation_value,
        net_product_sales_value,
        is_cancellation,
        has_product_sale,
        has_product_cancellation
    FROM fact_orders
    WHERE has_product_sale = TRUE
    ORDER BY gross_product_sales_value DESC
    LIMIT 20
    """
).df()

display(orders_examples)

,Invoice,order_datetime,customer_id,country,line_count,distinct_stockcodes,product_sale_quantity,gross_product_sales_value,product_cancellation_value,net_product_sales_value,is_cancellation,has_product_sale,has_product_cancellation
0,581483,2011-12-09 09:15:00,16446,United Kingdom,1,1,80995.0,168469.60,0.0,168469.60,False,True,False
1,541431,2011-01-18 10:01:00,12346,United Kingdom,1,1,74215.0,77183.60,0.0,77183.60,False,True,False
2,574941,2011-11-07 17:42:00,<NA>,United Kingdom,101,101,14149.0,52940.94,0.0,52940.94,False,True,False
3,576365,2011-11-14 17:55:00,<NA>,United Kingdom,99,99,13956.0,50653.91,0.0,50653.91,False,True,False
4,533027,2010-11-15 16:02:00,<NA>,United Kingdom,111,111,13387.0,49844.99,0.0,49844.99,False,True,False
5,531516,2010-11-08 16:45:00,<NA>,United Kingdom,115,115,12410.0,45332.97,0.0,45332.97,False,True,False
6,493819,2010-01-07 12:34:00,14156,EIRE,94,94,25018.0,44051.60,0.0,44051.60,False,True,False
7,556444,2011-06-10 15:28:00,15098,United Kingdom,1,1,60.0,38970.00,0.0,38970.00,False,True,False
8,524181,2010-09-27 16:59:00,17450,United Kingdom,14,14,8172.0,33167.80,0.0,33167.80,False,True,False
9,567423,2011-09-20 11:05:00,17450,United Kingdom,12,12,12572.0,31698.16,0.0,31698.16,False,True,False


In [16]:
# Exemples de factures d'annulation produit

cancellation_examples = connection.sql(
    """
    SELECT
        Invoice,
        order_datetime,
        customer_id,
        country,
        line_count,
        product_cancellation_quantity,
        product_cancellation_value,
        is_cancellation,
        has_product_cancellation
    FROM fact_orders
    WHERE has_product_cancellation = TRUE
    ORDER BY product_cancellation_value
    LIMIT 20
    """
).df()

display(cancellation_examples)

,Invoice,order_datetime,customer_id,country,line_count,product_cancellation_quantity,product_cancellation_value,is_cancellation,has_product_cancellation
0,C581484,2011-12-09 09:27:00,16446,United Kingdom,1,-80995.0,-168469.60,True,True
1,C541433,2011-01-18 10:17:00,12346,United Kingdom,1,-74215.0,-77183.60,True,True
2,C550456,2011-04-18 13:08:00,15749,United Kingdom,5,-9014.0,-22998.40,True,True
3,C524235,2010-09-28 11:02:00,14277,France,45,-87167.0,-11880.84,True,True
4,C570556,2011-10-11 11:10:00,16029,United Kingdom,12,-6480.0,-11816.64,True,True
5,C513771,2010-06-28 13:49:00,16754,United Kingdom,14,-6584.0,-7500.32,True,True
6,C529352,2010-10-28 09:32:00,17450,United Kingdom,9,-2760.0,-6325.08,True,True
7,C508132,2010-05-13 10:54:00,12931,United Kingdom,11,-3564.0,-4815.80,True,True
8,C508134,2010-05-13 10:56:00,12931,United Kingdom,11,-3560.0,-4798.80,True,True
9,C527790,2010-10-19 11:12:00,12454,Spain,8,-2232.0,-4733.52,True,True


## Préparation de dim_customers

La dimension client sera construite pour les clients avec un `Customer ID`.

Les factures sans identifiant client sont conservées dans les tables de faits afin de permettre les analyses agrégées, mais elles ne sont pas attachés à un client individuel.


In [17]:
# Vue générale des clients

customer_overview = connection.sql(
    """
    SELECT
        COUNT(DISTINCT customer_id) AS identified_customers,

        SUM(
            CASE
                WHEN customer_id IS NULL THEN 1
                ELSE 0
            END
        ) AS invoices_without_customer,

        COUNT(
            DISTINCT CASE
                WHEN customer_id IS NOT NULL
                THEN Invoice
            END
        ) AS invoices_with_customer

    FROM fact_orders
    """
).df()

display(customer_overview)

,identified_customers,invoices_without_customer,invoices_with_customer
0,5942,8752.0,44876


In [18]:
# Prévisualisation client

customer_preview = connection.sql(
    """
    SELECT
        customer_id,

        MIN(order_datetime) AS first_activity_datetime,
        MAX(order_datetime) AS last_activity_datetime,

        COUNT(*) AS invoice_count,

        SUM(
            CASE
                WHEN has_product_sale THEN 1
                ELSE 0
            END
        ) AS purchase_invoice_count,

        SUM(
            CASE
                WHEN has_product_cancellation THEN 1
                ELSE 0
            END
        ) AS cancellation_invoice_count,

        COUNT(DISTINCT country) AS distinct_countries,

        ROUND(
            SUM(gross_product_sales_value),
            2
        ) AS gross_product_sales_value,

        ROUND(
            SUM(product_cancellation_value),
            2
        ) AS product_cancellation_value,

        ROUND(
            SUM(net_product_sales_value),
            2
        ) AS net_product_sales_value

    FROM fact_orders

    WHERE customer_id IS NOT NULL

    GROUP BY customer_id

    ORDER BY net_product_sales_value DESC

    LIMIT 20
    """
).df()

display(customer_preview)

,customer_id,first_activity_datetime,last_activity_datetime,invoice_count,purchase_invoice_count,cancellation_invoice_count,distinct_countries,gross_product_sales_value,product_cancellation_value,net_product_sales_value
0,18102,2009-12-01 09:24:00,2011-12-09 11:50:00,153,145.0,2.0,1,580987.04,-2578.40,578408.64
1,14646,2009-12-02 16:52:00,2011-12-08 12:12:00,164,145.0,7.0,1,526751.52,-3548.78,523202.74
2,14156,2009-12-01 12:30:00,2011-11-30 10:54:00,202,144.0,38.0,1,303256.43,-5294.87,297961.56
3,14911,2009-12-01 11:41:00,2011-12-08 15:54:00,510,373.0,93.0,1,272393.03,-13791.56,258601.47
4,17450,2010-09-27 16:59:00,2011-12-01 13:29:00,61,51.0,4.0,1,244944.25,-11140.34,233803.91
5,13694,2009-12-04 15:26:00,2011-12-06 09:32:00,164,143.0,15.0,1,195640.69,-5286.63,190354.06
6,17511,2009-12-02 10:52:00,2011-12-07 10:12:00,85,60.0,24.0,1,172132.87,-3628.43,168504.44
7,12415,2010-06-30 08:30:00,2011-11-15 14:22:00,33,24.0,4.0,1,144033.37,-926.35,143107.02
8,16684,2009-12-07 12:56:00,2011-12-05 14:06:00,65,55.0,9.0,1,147142.77,-5612.48,141530.29
9,15061,2009-12-01 12:18:00,2011-12-06 12:06:00,138,127.0,10.0,1,126387.57,-1405.44,124982.13


In [19]:
connection.close()

print("Connexion DuckDB fermée")

Connexion DuckDB fermée


In [20]:
connection = duckdb.connect(
    str(DATABASE_FILE)
)

print("Connexion DuckDB : OK")

Connexion DuckDB : OK


In [21]:
# Aperçu de dim_customers

customers_sample = connection.sql(
    """
    SELECT *
    FROM dim_customers
    ORDER BY first_activity_datetime
    LIMIT 20
    """
).df()

display(customers_sample)

,customer_id,first_activity_datetime,last_activity_datetime,invoice_count,distinct_countries,is_multi_country,latest_country
0,13085,2009-12-01 07:45:00,2011-07-05 12:11:00,10,1,False,United Kingdom
1,13078,2009-12-01 09:06:00,2011-12-08 18:48:00,95,1,False,United Kingdom
2,15362,2009-12-01 09:08:00,2010-09-17 10:37:00,2,1,False,United Kingdom
3,18102,2009-12-01 09:24:00,2011-12-09 11:50:00,153,1,False,United Kingdom
4,12682,2009-12-01 09:28:00,2011-12-06 10:00:00,53,1,False,France
5,18087,2009-12-01 09:43:00,2011-09-02 15:12:00,21,1,False,United Kingdom
6,13635,2009-12-01 09:46:00,2011-10-03 10:55:00,6,1,False,United Kingdom
7,14110,2009-12-01 09:50:00,2011-12-06 12:26:00,35,1,False,United Kingdom
8,12636,2009-12-01 09:55:00,2009-12-01 09:55:00,1,1,False,USA
9,17519,2009-12-01 09:57:00,2011-11-22 16:43:00,17,1,False,United Kingdom


In [22]:
# Clients observés dans plusieurs pays

multi_country_customers = connection.sql(
    """
    SELECT *
    FROM dim_customers
    WHERE is_multi_country = TRUE
    ORDER BY customer_id
    """
).df()

print(
    "Nombre de clients multi-pays :",
    len(multi_country_customers)
)

display(multi_country_customers)

Nombre de clients multi-pays : 13


,customer_id,first_activity_datetime,last_activity_datetime,invoice_count,distinct_countries,is_multi_country,latest_country
0,12370,2010-02-09 09:41:00,2011-10-19 14:51:00,7,2,True,Cyprus
1,12394,2011-05-06 14:01:00,2011-10-07 08:08:00,2,2,True,Denmark
2,12413,2010-11-02 13:26:00,2011-10-04 09:00:00,6,2,True,France
3,12417,2009-12-06 10:30:00,2011-12-06 14:52:00,27,2,True,Belgium
4,12422,2009-12-18 12:44:00,2011-09-05 09:48:00,21,2,True,Australia
5,12423,2010-08-31 09:31:00,2011-12-09 10:10:00,11,2,True,Belgium
6,12429,2010-06-11 13:28:00,2011-11-30 17:22:00,9,2,True,Denmark
7,12431,2010-01-15 09:04:00,2011-11-04 11:55:00,33,2,True,Australia
8,12449,2010-09-27 09:37:00,2011-11-17 13:46:00,6,2,True,Belgium
9,12455,2009-12-18 12:33:00,2011-09-27 15:31:00,10,2,True,Spain


In [24]:
connection.close()
print("Connexion DuckDB fermée")

Connexion DuckDB fermée


In [25]:
connection = duckdb.connect(
    str(DATABASE_FILE)
)

In [26]:
#  dim_products

products_sample = connection.sql(
    """
    SELECT *
    FROM dim_products
    ORDER BY stock_code
    LIMIT 20
    """
).df()

display(products_sample)

,stock_code,product_description,line_category,first_seen_datetime,last_seen_datetime,distinct_descriptions
0,10002,INFLATABLE POLITICAL GLOBE,product,2009-12-01 09:08:00,2011-04-28 15:05:00,1
1,10002R,ROBOT PENCIL SHARPNER,product,2009-12-02 14:43:00,2010-01-25 17:36:00,1
2,10080,GROOVY CACTUS INFLATABLE,product,2009-12-02 16:02:00,2011-11-21 17:04:00,2
3,10109,BENDY COLOUR PENCILS,product,2009-12-03 12:31:00,2010-02-04 13:47:00,1
4,10120,DOGGY RUBBER,product,2009-12-01 16:17:00,2011-12-04 13:15:00,2
5,10123C,HEARTS WRAPPING TAPE,product,2009-12-01 14:28:00,2011-07-15 15:05:00,1
6,10123G,ARMY CAMO WRAPPING TAPE,product,2009-12-01 14:28:00,2011-04-08 11:13:00,1
7,10124A,SPOTS ON RED BOOKCOVER TAPE,product,2010-02-11 18:13:00,2011-11-06 13:00:00,1
8,10124C,NaN,product,2010-05-11 12:51:00,2010-05-11 12:51:00,0
9,10124G,ARMY CAMO BOOKCOVER TAPE,product,2010-05-13 16:49:00,2011-11-06 13:00:00,1


In [27]:
# Exemples de produits ayant plusieurs descriptions observées

products_multiple_descriptions = connection.sql(
    """
    SELECT
        stock_code,
        product_description,
        line_category,
        distinct_descriptions
    FROM dim_products
    WHERE distinct_descriptions > 1
    ORDER BY distinct_descriptions DESC
    LIMIT 20
    """
).df()

display(products_multiple_descriptions)

,stock_code,product_description,line_category,distinct_descriptions
0,20713,JUMBO BAG OWLS,product,9
1,22423,REGENCY CAKESTAND 3 TIER,product,7
2,22734,SET OF 6 RIBBONS VINTAGE CHRISTMAS,product,7
3,21181,PLEASE ONE PERSON METAL SIGN,product,7
4,23084,RABBIT NIGHT LIGHT,product,7
5,47566B,TEA TIME PARTY BUNTING,product,6
6,85175,CACTI T-LIGHT CANDLES,product,6
7,21830,ASSORTED CREEPY CRAWLIES,product,6
8,22719,GUMBALL MONOCHROME COAT RACK,product,6
9,22501,PICNIC BASKET WICKER LARGE,product,5


In [29]:
connection.close()

In [30]:
connection.close()

print("Connexion DuckDB fermée")

Connexion DuckDB fermée
